### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: SahilDargar_23244_Resume.pdf
  ✓ Loaded 1 pages

Processing: Sahil_D.pdf
  ✓ Loaded 1 pages

Total documents loaded: 2


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-07-01T16:50:10+00:00', 'author': '', 'keywords': '', 'moddate': '2026-07-01T16:50:10+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\SahilDargar_23244_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'SahilDargar_23244_Resume.pdf', 'file_type': 'pdf'}, page_content='Sahil Dargar +91-8882386329\nRoll No.:23244 sahildargar@gmail.com\nB.Tech - Electronics and Communication Engineering Github\nIndian Institute Of Information Technology, Una linkedin.com/in/dargar1726\nEducation\nDegree/Certificate Institute/Board CGPA/Percentage Year\nB.Tech Indian Institute of Information Technology, Una 7.53 (Current) 2023-Present\nSenior Secondary CBSE Board 93.2% 2022\nSecondary CBSE Board 94.8% 2020\nProjects\n• MindEase Jul 2025

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 2 documents into 9 chunks

Example chunk:
Content: Sahil Dargar +91-8882386329
Roll No.:23244 sahildargar@gmail.com
B.Tech - Electronics and Communication Engineering Github
Indian Institute Of Information Technology, Una linkedin.com/in/dargar1726
Ed...
Metadata: {'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-07-01T16:50:10+00:00', 'author': '', 'keywords': '', 'moddate': '2026-07-01T16:50:10+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\SahilDargar_23244_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'SahilDargar_23244_Resume.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-07-01T16:50:10+00:00', 'author': '', 'keywords': '', 'moddate': '2026-07-01T16:50:10+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\SahilDargar_23244_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'SahilDargar_23244_Resume.pdf', 'file_type': 'pdf'}, page_content='Sahil Dargar +91-8882386329\nRoll No.:23244 sahildargar@gmail.com\nB.Tech - Electronics and Communication Engineering Github\nIndian Institute Of Information Technology, Una linkedin.com/in/dargar1726\nEducation\nDegree/Certificate Institute/Board CGPA/Percentage Year\nB.Tech Indian Institute of Information Technology, Una 7.53 (Current) 2023-Present\nSenior Secondary CBSE Board 93.2% 2022\nSecondary CBSE Board 94.8% 2020\nProjects\n• MindEase Jul 2025

### embedding And vectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3081.89it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [11]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [12]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-07-01T16:50:10+00:00', 'author': '', 'keywords': '', 'moddate': '2026-07-01T16:50:10+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\SahilDargar_23244_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'SahilDargar_23244_Resume.pdf', 'file_type': 'pdf'}, page_content='Sahil Dargar +91-8882386329\nRoll No.:23244 sahildargar@gmail.com\nB.Tech - Electronics and Communication Engineering Github\nIndian Institute Of Information Technology, Una linkedin.com/in/dargar1726\nEducation\nDegree/Certificate Institute/Board CGPA/Percentage Year\nB.Tech Indian Institute of Information Technology, Una 7.53 (Current) 2023-Present\nSenior Secondary CBSE Board 93.2% 2022\nSecondary CBSE Board 94.8% 2020\nProjects\n• MindEase Jul 2025

In [13]:
## Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

## store in the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 9 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.51it/s]

Generated embeddings with shape: (9, 384)
Adding 9 documents to vector store...
Successfully added 9 documents to vector store
Total documents in collection: 9


### Retriever Pipeline From VectorStore

In [29]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                print("Distances:", distances)
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "distance": distance,
                        "rank": i + 1
                    })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [30]:
rag_retriever

In [32]:
rag_retriever.retrieve("Java")

Retrieving documents for query: 'Java'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.03it/s]

Generated embeddings with shape: (1, 384)
Distances: [1.2118743658065796, 1.261211633682251, 1.323047399520874, 1.4817619323730469, 1.6110882759094238]
Retrieved 5 documents (after filtering)


[{'id': 'doc_761983d3_8',
  'content': 'Event Organizer, E-Summit: Contributed to organizing the institute’s first E-Summit.\nAchievements\nAcademics: Solved 450+ DSA questions (Java) on LeetCode and GFG .\nSports: Played F ootball at District Level , Runner-up in Chess at school competition.',
  'metadata': {'total_pages': 1,
   'title': "John Doe's CV",
   'page': 0,
   'author': 'John Doe',
   'keywords': '',
   'doc_index': 8,
   'creator': 'LaTeX with RenderCV',
   'subject': '',
   'moddate': '2026-07-01T16:36:51+00:00',
   'source': '..\\data\\pdf\\Sahil_D.pdf',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0',
   'content_length': 250,
   'source_file': 'Sahil_D.pdf',
   'trapped': '/False',
   'file_type': 'pdf',
   'producer': 'pdfTeX-1.40.26',
   'page_label': '1',
   'creationdate': '2026-07-01T16:36:51+00:00'},
  'distance': 1.2118743658065796,
  'rank': 1},
 {'id': 'doc_1e252404_5',
  'content': 'SAHIL DARGAR\n

In [33]:
rag_retriever.retrieve("React")

Retrieving documents for query: 'React'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.35it/s]

Generated embeddings with shape: (1, 384)
Distances: [1.5495386123657227, 1.6585910320281982, 1.663187026977539, 1.7067644596099854, 1.7976118326187134]
Retrieved 5 documents (after filtering)


[{'id': 'doc_ff75e780_2',
  'content': 'time, and utilized ImageKit for efficient cloud-based image optimization and delivery.\n• Jobify Nov. 2024 - Dec. 2024\nPracticum, IIIT Una Website\n– Engineered a scalable MERN-based job portal capable of handling100+ concurrent users, providing secure au-\nthentication, job discovery, and resume management through real-time CRUD operations.\n– Constructed and optimized 10+ RESTful APIs for users, jobs, companies, and applications, leveraging JWT\nauthentication and role-based authorization.\n– Optimized state management and file processing by leveraging Redux Toolkit and Multer, resulting in30% faster\nresume and profile update workflows.\n– Designed a responsive and accessible user interface using React (Vite) and Shadcn UI, enabling advanced job\nfiltering, application tracking, and an admin dashboard overseeing50+ job listings.\nTechnical Skills\n• Programming:Python, C, Java, Javascript, Typescript\n• Frameworks:ReactJS, NodeJS, ExpressJS, 

### RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [40]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage